<a href="https://colab.research.google.com/github/JJcoders00/slm/blob/main/JJ_Coders_AI_General_Intelligence_Pipeline-notebook-3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JJ Coders - General Language Model (Stage 3)
### An Open-Domain Foundational Architecture and Training Pipeline

This notebook implements a complete pipeline for training a Small Language Model (SLM) from scratch, incorporating:
- **Architecture:** Recurrent Transformer with RMSNorm, Rotary Position Embeddings (RoPE), SwiGLU activations, and weight-tied recurrent depth (~30M parameters).
- **Data:** Multi-domain corpus spanning instruction dialogues, general knowledge, and task reasoning.
- **Persistence:** Automated model state checkpointing and tokenizer export to Google Drive (`JJ_AI_Project`).

## 1. Environment Configuration and Storage Setup
Initializes the environment, verifies compute hardware, and connects Google Drive for storage.

In [1]:
import os
import torch

print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("GPU not detected. Please select T4 GPU under Runtime > Change runtime type.")

from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/JJ_AI_Project'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Workspace directory: {SAVE_DIR}")

PyTorch Version: 2.11.0+cu128
Device: Tesla T4
VRAM: 15.64 GB
Mounted at /content/drive
Workspace directory: /content/drive/MyDrive/JJ_AI_Project


## 2. Multi-Domain Dataset Streaming
Streams conversational instruction data and domain knowledge into a consolidated text corpus.

In [2]:
!pip install -q tokenizers datasets

from datasets import load_dataset

DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)
general_corpus_path = os.path.join(DATA_DIR, 'general_corpus.txt')

print("Compiling multi-domain training corpus...")

with open(general_corpus_path, 'w', encoding='utf-8') as f_out:
    # 1. Open-Domain Assistant Dialogues
    print("1/3 Processing conversational dialogues...")
    try:
        chat_stream = load_dataset('HuggingFaceH4/ultrachat_200k', split='train_sft', streaming=True)
        count = 0
        for item in chat_stream:
            messages = item.get('messages', [])
            if len(messages) >= 2:
                u_msg = messages[0].get('content', '').strip()
                a_msg = messages[1].get('content', '').strip()
                if u_msg and a_msg:
                    f_out.write(f'<user> {u_msg[:500]} <bot> {a_msg[:800]} <|endoftext|>\n')
                    count += 1
                    if count >= 8000:
                        break
        print(f"    Integrated {count} dialogues.")
    except Exception as e:
        print(f"    Dialogue stream note: {e}")

    # 2. General Text and Knowledge Articles
    print("2/3 Processing general knowledge articles...")
    try:
        story_stream = load_dataset('roneneldan/TinyStories', split='train', streaming=True)
        count = 0
        for item in story_stream:
            text = item.get('text', '').strip()
            if text:
                f_out.write(text + '\n<|endoftext|>\n')
                count += 1
                if count >= 12000:
                    break
        print(f"    Integrated {count} articles.")
    except Exception as e:
        print(f"    Article stream note: {e}")

    # 3. Structured Tasks and Reasoning Patterns
    print("3/3 Processing structured task patterns...")
    task_templates = [
        ('Help me outline a science project on solar energy.', 'Here is a structured project outline:\n1. Introduction and Renewable Energy Concepts\n2. Photovoltaic Conversion Mechanisms\n3. Applications in Residential and Industrial Contexts\n4. Economic and Environmental Trade-offs\n5. Summary and Future Directions'),
        ('How do I plan my study schedule for exams?', 'A structured study plan consists of the following steps:\n1. Prioritize subjects based on syllabus weight and exam dates.\n2. Allocate 45-minute focused blocks followed by brief intervals.\n3. Dedicate high-energy hours to analytical problem solving.\n4. Conclude with summary review and practice problems.'),
        ('Write a Python function to check if a number is prime.', 'def is_prime(n):\n    if n < 2:\n        return False\n    for i in range(2, int(n**0.5) + 1):\n        if n % i == 0:\n            return False\n    return True'),
        ('What is the difference between a planet and a star?', 'A star is a self-luminous celestial body undergoing nuclear fusion, whereas a planet is a non-luminous body that orbits a star.'),
        ('Give me 3 tips for effective essay writing.', '1. Formulate a precise thesis statement in the introduction.\n2. Support each body paragraph with concrete evidence.\n3. Maintain coherent transitions and synthesize findings in the conclusion.')
    ] * 200
    for u_q, b_a in task_templates:
        f_out.write(f'<user> {u_q} <bot> {b_a} <|endoftext|>\n')

corpus_size_mb = os.path.getsize(general_corpus_path) / 1e6
print(f"Corpus preparation complete: {corpus_size_mb:.2f} MB")

Compiling multi-domain training corpus...
1/3 Processing conversational dialogues...


README.md:   0%|          | 0.00/3.90k [00:00<?, ?B/s]

    Integrated 8000 dialogues.
2/3 Processing general knowledge articles...


README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

    Integrated 12000 articles.
3/3 Processing structured task patterns...
Corpus preparation complete: 19.53 MB


## 3. Byte-Pair Encoding (BPE) Tokenizer Training
Trains a custom tokenizer with an 8,192 token vocabulary on the prepared corpus.

In [3]:
from tokenizers import ByteLevelBPETokenizer

TOKENIZER_DIR = os.path.join(SAVE_DIR, 'jj_general_tokenizer')
os.makedirs(TOKENIZER_DIR, exist_ok=True)

tokenizer = ByteLevelBPETokenizer()
tokenizer.train(
    files=[general_corpus_path],
    vocab_size=8192,
    min_frequency=2,
    special_tokens=['<pad>', '<s>', '</s>', '<unk>', '<|endoftext|>', '<user>', '<bot>']
)

tokenizer.save_model(TOKENIZER_DIR)
print(f"Tokenizer saved to: {TOKENIZER_DIR}")

Tokenizer saved to: /content/drive/MyDrive/JJ_AI_Project/jj_general_tokenizer


## 4. Binary Tokenization and Memory Mapping
Encodes the corpus into an unsigned 16-bit integer binary array for memory-mapped streaming during training.

In [4]:
import numpy as np

bin_path = os.path.join(DATA_DIR, 'general_train.bin')
if os.path.exists(bin_path):
    os.remove(bin_path)

print("Encoding corpus into binary stream...")
total_tokens = 0
chunk_size = 2000
lines_buffer = []

with open(general_corpus_path, 'r', encoding='utf-8') as f_in, open(bin_path, 'wb') as f_bin:
    for line in f_in:
        lines_buffer.append(line)
        if len(lines_buffer) >= chunk_size:
            text_chunk = ''.join(lines_buffer)
            encoded = tokenizer.encode(text_chunk).ids
            arr = np.array(encoded, dtype=np.uint16)
            f_bin.write(arr.tobytes())
            total_tokens += len(arr)
            lines_buffer = []

    if lines_buffer:
        text_chunk = ''.join(lines_buffer)
        encoded = tokenizer.encode(text_chunk).ids
        arr = np.array(encoded, dtype=np.uint16)
        f_bin.write(arr.tobytes())
        total_tokens += len(arr)

print(f"Total tokens compiled: {total_tokens:,}")
print(f"Binary file path: {bin_path} ({os.path.getsize(bin_path) / 1e6:.2f} MB)")

Encoding corpus into binary stream...
Total tokens compiled: 4,814,494
Binary file path: /content/data/general_train.bin (9.63 MB)


## 5. Model Architecture Specification
Defines the Transformer architecture with Rotary Position Embeddings (RoPE), RMSNorm, SwiGLU Feed-Forward Networks, and Recurrent Depth.

In [5]:
import math
import torch.nn as nn
import torch.nn.functional as F

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

def precompute_rope_freqs(dim: int, max_seq_len: int, theta: float = 10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    t = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    return torch.polar(torch.ones_like(freqs), freqs)

def apply_rotary_emb(xq, xk, freqs_cis):
    xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))
    freqs_cis = freqs_cis[:xq.shape[1], :].to(xq.device).view(1, xq.shape[1], 1, -1)
    xq_out = torch.view_as_real(xq_ * freqs_cis).flatten(3)
    xk_out = torch.view_as_real(xk_ * freqs_cis).flatten(3)
    return xq_out.type_as(xq), xk_out.type_as(xk)

class SwiGLUMLP(nn.Module):
    def __init__(self, dim: int, hidden_dim: int):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden_dim, bias=False)
        self.w2 = nn.Linear(dim, hidden_dim, bias=False)
        self.w3 = nn.Linear(hidden_dim, dim, bias=False)

    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))

class TransformerBlock(nn.Module):
    def __init__(self, dim: int, n_heads: int):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)
        self.norm1 = RMSNorm(dim)
        self.norm2 = RMSNorm(dim)
        self.mlp = SwiGLUMLP(dim, int(dim * 2.67))

    def forward(self, x, freqs_cis):
        B, S, D = x.shape
        norm_x = self.norm1(x)
        q = self.q_proj(norm_x).view(B, S, self.n_heads, self.head_dim)
        k = self.k_proj(norm_x).view(B, S, self.n_heads, self.head_dim)
        v = self.v_proj(norm_x).view(B, S, self.n_heads, self.head_dim)
        q, k = apply_rotary_emb(q, k, freqs_cis)
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
        attn_out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, S, D)
        h = x + self.out_proj(attn_out)
        return h + self.mlp(self.norm2(h))

class JJGeneralModel(nn.Module):
    def __init__(self, vocab_size=8192, dim=384, n_heads=6, n_layers=6, recurrent_steps=2, max_seq_len=512):
        super().__init__()
        self.recurrent_steps = recurrent_steps
        self.embed = nn.Embedding(vocab_size, dim)
        self.blocks = nn.ModuleList([TransformerBlock(dim, n_heads) for _ in range(n_layers)])
        self.final_norm = RMSNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)
        self.embed.weight = self.lm_head.weight
        self.register_buffer('freqs_cis', precompute_rope_freqs(dim // n_heads, max_seq_len), persistent=False)

    def forward(self, input_ids, targets=None):
        x = self.embed(input_ids)
        for _ in range(self.recurrent_steps):
            for block in self.blocks:
                x = block(x, self.freqs_cis)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=150, temperature=0.7, top_k=40, stop_token_id=None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = input_ids if input_ids.size(1) <= 512 else input_ids[:, -512:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat((input_ids, idx_next), dim=1)
            if stop_token_id is not None and idx_next.item() == stop_token_id:
                break
        return input_ids

print("Model architecture initialized.")

Model architecture initialized.


## 6. Training Engine and State Checkpointing
Executes the training loop with mixed precision and auto-saves checkpoints to Google Drive.

In [6]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
GENERAL_CHECKPOINT_PATH = os.path.join(SAVE_DIR, 'jj_general_model_checkpoint.pt')

model = JJGeneralModel(
    vocab_size=8192,
    dim=384,
    n_heads=6,
    n_layers=6,
    recurrent_steps=2,
    max_seq_len=512
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Physical Parameters: {total_params / 1e6:.2f}M | Effective Depth: 12 Layers")

def get_batch(bin_file, batch_size=32, seq_len=256):
    data = np.memmap(bin_file, dtype=np.uint16, mode='r')
    ix = torch.randint(len(data) - seq_len - 1, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i+seq_len]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+seq_len]).astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=4e-4, weight_decay=0.01)
scaler = torch.cuda.amp.GradScaler()

start_step = 0
if os.path.exists(GENERAL_CHECKPOINT_PATH):
    print("Loading existing checkpoint from Google Drive...")
    ckpt = torch.load(GENERAL_CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scaler.load_state_dict(ckpt['scaler_state_dict'])
    start_step = ckpt['step'] + 1
    print(f"Resumed from step {start_step} (Saved Loss: {ckpt['loss']:.4f})")
else:
    print("Starting new training run.")

max_steps = 2500
eval_interval = 250
save_interval = 500

model.train()
print(f"Executing training for {max_steps} steps...")

for step in range(start_step, max_steps):
    xb, yb = get_batch(bin_path, batch_size=32, seq_len=256)
    optimizer.zero_grad(set_to_none=True)

    with torch.cuda.amp.autocast(dtype=torch.float16):
        logits, loss = model(xb, targets=yb)

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()

    if (step + 1) % eval_interval == 0 or step == max_steps - 1:
        print(f"Step [{step+1}/{max_steps}] | Loss: {loss.item():.4f}")

    if (step + 1) % save_interval == 0 or step == max_steps - 1:
        torch.save({
            'step': step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'loss': loss.item()
        }, GENERAL_CHECKPOINT_PATH)
        print(f"--> Checkpoint saved to Google Drive at step {step+1}")

print("Training run complete.")

Physical Parameters: 13.77M | Effective Depth: 12 Layers


/tmp/ipykernel_1726/242027835.py:24: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_1726/242027835.py:49: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):


Starting new training run.
Executing training for 2500 steps...
Step [250/2500] | Loss: 4.5746
Step [500/2500] | Loss: 4.0514
--> Checkpoint saved to Google Drive at step 500
Step [750/2500] | Loss: 3.7355
Step [1000/2500] | Loss: 3.1934
--> Checkpoint saved to Google Drive at step 1000
Step [1250/2500] | Loss: 3.3028
Step [1500/2500] | Loss: 3.0307
--> Checkpoint saved to Google Drive at step 1500
Step [1750/2500] | Loss: 3.1312
Step [2000/2500] | Loss: 2.6629
--> Checkpoint saved to Google Drive at step 2000
Step [2250/2500] | Loss: 2.7433
Step [2500/2500] | Loss: 2.8572
--> Checkpoint saved to Google Drive at step 2500
Training run complete.


## 7. Inference and Evaluation
Evaluates model generation across diverse prompt domains.

In [7]:
def ask_jj_ai(user_prompt):
    formatted_prompt = f'<user> {user_prompt} <bot>'
    input_ids = torch.tensor([tokenizer.encode(formatted_prompt).ids], device=device)
    end_id = tokenizer.token_to_id('<|endoftext|>')

    generated_ids = model.generate(
        input_ids,
        max_new_tokens=150,
        temperature=0.7,
        top_k=40,
        stop_token_id=end_id
    )

    output_text = tokenizer.decode(generated_ids[0].tolist())
    reply = output_text.split('<bot>')[-1].replace('<|endoftext|>', '').strip()
    return reply

prompts = [
    'I have a science assignment on the solar system tomorrow. Can you help me outline the main planets?',
    'How do you define artificial intelligence in simple terms?',
    'Write a short Python function to calculate the square of a number.',
    'Give me 3 tips for effective time management.'
]

print("=== JJ CODERS MODEL INFERENCE ===\n")
for p in prompts:
    print(f"User: {p}")
    response = ask_jj_ai(p)
    print(f"JJ AI: {response}\n")
    print('-' * 50)

=== JJ CODERS MODEL INFERENCE ===

User: I have a science assignment on the solar system tomorrow. Can you help me outline the main planets?
JJ AI: I have a science assignment on the solar system tomorrow. Can you help me outline the main planets?  I do not have access to the following steps:

1. Set the following command: a program-friendly interface design is the process to manage and ensure they take to connect the system's performance. This can be achieved by creating a user-friendly interface that allows users to store angle or browser.

2. Customer Solution: The application should have a clean and user-friendly interface that allows users to create and implement in their own communication skills. 

3. Implement a new screen: The application should have an intuitive user experience, with the user interface, and the user interface for the website. 

4. Use the structure of the application to create the interface and color the website. The

------------------------------------------